# 03 · Querying and Settings

> **Module 7 · Python Client Library · Lecture 7.4**

In this notebook you will:

- Run queries and understand the job lifecycle
- Return results as rows, DataFrames, and pages
- Use typed parameters, byte caps, and dry runs
- Introspect a job for cost, timing, and cache status
- Write DML and stream rows

---

## Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")
from bq_client import get_client, run_query, run_to_dataframe, estimate_cost

from google.cloud import bigquery

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "YOUR_PROJECT_ID")
client = get_client(project=PROJECT_ID)
print(f"Project: {client.project}")

## 1. The job lifecycle

```
client.query(sql)   -> QueryJob      (submitted, running in the background)
job.result()        -> RowIterator   (blocks until the query finishes)
```

`client.query()` is **asynchronous**. You get results only when you call
`.result()`.

In [ ]:
sql = """
    SELECT
      continent_name,
      COUNT(*) AS country_count
    FROM `bigquery-public-data.utility_us.country_code_iso`
    GROUP BY continent_name
    ORDER BY country_count DESC
"""

job = client.query(sql)
print(f"Submitted: {job.job_id}")
print(f"State:     {job.state}")

rows = job.result()
print(f"\nDone. {len(rows)} rows:")
for row in rows:
    print(f"  {row.continent_name:20} {row.country_count:>5}")

## 2. Three ways to get results

| Method | Use when |
|--------|----------|
| `list(job.result())` | Small results, row access |
| `job.to_dataframe()` | Exploration and analysis |
| `job.result().pages` | Large results — process page by page |

In [ ]:
# 1. Rows
job = client.query(sql)
rows = list(job.result())
print(f"Rows: {len(rows)}")
print(f"First: {dict(rows[0])}")

# 2. DataFrame
job = client.query(sql)
df = job.to_dataframe(progress_bar_type=None, create_bqstorage_client=False)
display(df)
print(df.dtypes)

In [ ]:
# 3. Paginated — for results too large for memory
job = client.query(sql)
frames = []
for page in job.result().pages:
    frames.append(page.to_dataframe())

import pandas as pd
paged = pd.concat(frames, ignore_index=True)
print(f"Pages: {len(frames)}, Rows: {len(paged)}")
display(paged)

In [ ]:
# A safe paginator that never loads everything at once


def iter_rows(sql, page_size=10_000):
    \"\"\"Yield rows in pages without materialising the whole result.\"\"\"
    job = client.query(sql)
    rows = job.result()
    while True:
        page = rows.fetchmany(page_size)
        if not page:
            break
        yield from page


count = 0
for row in iter_rows(sql):
    count += 1
print(f"Iterated {count} rows")

## 3. Parameters — always, with explicit types

`ScalarQueryParameter(name, type, value)` — the type is required. Inferring it from
the Python value produces surprises.

In [ ]:
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter("continent", "STRING", "Europe"),
        bigquery.ScalarQueryParameter("min_count", "INT64", 10),
    ]
)

sql_p = """
    SELECT country_code, country_name
    FROM `bigquery-public-data.utility_us.country_code_iso`
    WHERE continent_name = @continent
    LIMIT @min_count
"""

df = client.query(sql_p, job_config=job_config).to_dataframe(
    progress_bar_type=None, create_bqstorage_client=False
)
display(df)

In [ ]:
# Array parameters
job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ArrayQueryParameter(
            "continents", "STRING", ["Europe", "Africa", "Asia"]
        ),
    ]
)

sql_a = """
    SELECT continent_name, COUNT(*) AS n
    FROM `bigquery-public-data.utility_us.country_code_iso`
    WHERE continent_name IN UNNEST(@continents)
    GROUP BY continent_name
"""

df = client.query(sql_a, job_config=job_config).to_dataframe(
    progress_bar_type=None, create_bqstorage_client=False
)
display(df)

## 4. Cost control — the two tools

### 4a. Dry run — free and exact

### 4b. `maximum_bytes_billed` — the query fails instead of billing

> 💡 A query that fails is free. A query that runs without a cap is not.

In [ ]:
BIG_SQL = "SELECT * FROM `bigquery-public-data.world_bank.country_population`"

est = estimate_cost(BIG_SQL)
print(f"Bytes:       {est['bytes_processed']:,}")
print(f"TiB:         {est['tib']:.4f}")
print(f"Est. cost:   ${est['estimated_cost_usd']:.4f}")
print("\n(This dry run cost nothing and took about a second.)")

In [ ]:
# Enforce a byte cap — the query FAILS if it would exceed this
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000)  # 1 MB

try:
    client.query(BIG_SQL, job_config=job_config).result()
    print("Query succeeded (unexpected with a 1 MB cap)")
except Exception as exc:
    print(f"✅ Blocked as intended:\n   {str(exc)[:180]}")

## 5. Job introspection

These fields are your best debugging tools.

In [ ]:
job = client.query(sql)
rows = job.result()

print(f"job_id:            {job.job_id}")
print(f"query_id:          {job.query_id}")
print(f"state:             {job.state}")
print(f"created:           {job.created}")
print(f"started:           {job.started}")
print(f"ended:             {job.ended}")
print(f"location:          {job.location}")
print(f"bytes processed:   {job.total_bytes_processed:,}")
print(f"bytes billed:      {job.total_bytes_billed:,}")
print(f"cache_hit:         {job.cache_hit}")
print(f"dml rows affected: {job.num_dml_affected_rows}")
print(f"parent job:        {job.parent_job_id}")

In [ ]:
# The execution plan
if job.query_plan:
    for step in job.query_plan:
        print(f"  {step.get('name', '?'):<28} {step.get('status', '?'):<12} "
              f"inputs={step.get('parallelInputs', '?')}")
else:
    print("No query plan available (typical for very small queries).")

# The full job metadata
print(f"\nLabels: {job.labels}")
print(f"Destination: {job.destination}")

## 6. The query cache

A repeated query within 24 hours is served from the cache and bills **zero bytes**.

In [ ]:
CACHE_SQL = "SELECT COUNT(*) AS n FROM `bigquery-public-data.utility_us.country_code_iso`"

for i in (1, 2):
    job = client.query(CACHE_SQL)
    job.result()
    print(f"Run {i}: cache_hit={job.cache_hit}, bytes_billed={job.total_bytes_billed:,}")

## 7. DML — the same API, different verbs

> ⚠️ **BigQuery DML rewrites whole blocks.** `UPDATE` on a million-row table is slow
> and expensive. Prefer `MERGE` into a partition, or recompute and replace the
> partition.

In [ ]:
# INSERT
job = client.query(
    f"""
    INSERT INTO `{PROJECT_ID}.curated.orders`
      (order_id, customer_id, order_date, total_amount, status)
    VALUES ('o9999', 'c0001', DATE '2026-01-16', 99.99, 'active')
    """
)
job.result()
print(f"INSERT affected {job.num_dml_affected_rows} row")

# UPDATE
job = client.query(
    f"""
    UPDATE `{PROJECT_ID}.curated.orders`
    SET status = 'cancelled'
    WHERE order_id = 'o9999'
    """
)
job.result()
print(f"UPDATE affected {job.num_dml_affected_rows} row")

# DELETE
job = client.query(
    f"""
    DELETE FROM `{PROJECT_ID}.curated.orders`
    WHERE order_id = 'o9999'
    """
)
job.result()
print(f"DELETE affected {job.num_dml_affected_rows} row")

In [ ]:
# MERGE — the upsert. Match on the partition key.
MERGE_SQL = f"""
MERGE `{PROJECT_ID}.curated.orders` AS t
USING (
  SELECT 'o0001' AS order_id, 'c0001' AS customer_id, DATE '2026-01-15' AS order_date,
         123.45 AS total_amount, 'active' AS status
) AS s
ON t.order_id = s.order_id AND t.order_date = s.order_date
WHEN MATCHED THEN UPDATE SET t.total_amount = s.total_amount, t.status = s.status
WHEN NOT MATCHED THEN INSERT (order_id, customer_id, order_date, total_amount, status)
VALUES (s.order_id, s.customer_id, s.order_date, s.total_amount, s.status)
"""

job = client.query(MERGE_SQL)
job.result()
print(f"MERGE affected {job.num_dml_affected_rows} rows")

## 8. Streaming inserts

The first 1 GB per table per day is free. After that it is $0.01 per 200 MB.

> ⚠️ **Do not stream-insert millions of rows.** `insert_rows_json` sends up to 1,000
> rows per request. For bulk data, use a load job.

In [ ]:
events_schema = [
    bigquery.SchemaField("event_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("user_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("event_type", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("event_ts", "TIMESTAMP", mode="REQUIRED"),
]

events_table = bigquery.Table(f"{PROJECT_ID}.raw.events", schema=events_schema)
events_table = client.create_table(events_table, exists_ok=True)

table_ref = client.dataset("raw").table("events")

rows = [
    {
        "event_id": f"e{i:06d}",
        "user_id": f"u{i % 100:04d}",
        "event_type": "click",
        "event_ts": "2026-01-15T10:00:00Z",
    }
    for i in range(1000)
]

errors = client.insert_rows_json(table_ref, rows)
if errors:
    print(f"❌ {len(errors)} failed rows")
    print("First error:", errors[0]["errors"])
else:
    print(f"✅ Streamed {len(rows)} rows")

In [ ]:
# Query the streamed data
df = run_to_dataframe(
    f"""
    SELECT event_type, COUNT(*) AS n, COUNT(DISTINCT user_id) AS users
    FROM `{PROJECT_ID}.raw.events`
    GROUP BY event_type
    """
)
display(df)

## 9. Write results to a table

A query with a `destination` **never uses the cache**.

In [ ]:
job_config = bigquery.QueryJobConfig(
    destination=f"{PROJECT_ID}.mart.continent_counts",
    write_disposition="WRITE_TRUNCATE",
)

job = client.query(
    """
    SELECT continent_name, COUNT(*) AS country_count
    FROM `bigquery-public-data.utility_us.country_code_iso`
    GROUP BY continent_name
    """,
    job_config=job_config,
)
job.result()
print(f"✓ Wrote {job.destination}")
print(f"  cache_hit={job.cache_hit} (always False with a destination)")

In [ ]:
# Export to Parquet in Cloud Storage
job_config = bigquery.ExportJobConfig(
    destination="gs://YOUR_PROJECT_ID-bq-course/exports/continents_*.parquet",
    destination_format="PARQUET",
    use_avro_logical_types=True,
)

try:
    job = client.query(
        "SELECT * FROM `bigquery-public-data.utility_us.country_code_iso`",
        job_config=job_config,
    )
    job.result()
    print("✓ Exported")
except Exception as exc:
    print(f"Export failed (bucket may not exist): {str(exc)[:150]}")

## 10. Build a reusable query helper

Never scatter `client.query()` across a codebase. Wrap it once.

In [ ]:
QUERY_HELPER_SOURCE = '''
"""
bq_query.py — a thin, safe wrapper over client.query().
"""
from __future__ import annotations

import datetime
import logging
from typing import Any

from google.cloud import bigquery
from google.cloud.exceptions import GoogleCloudError

log = logging.getLogger(__name__)
PRICE_PER_TIB = 6.25


def _sql_type(value: Any) -> str:
    if isinstance(value, bool):
        return "BOOL"
    if isinstance(value, int):
        return "INT64"
    if isinstance(value, float):
        return "FLOAT64"
    if isinstance(value, datetime.datetime):
        return "TIMESTAMP"
    if isinstance(value, datetime.date):
        return "DATE"
    return "STRING"


def run(sql, params=None, client=None, max_bytes_billed=10 * 1024**4, label=""):
    \"\"\"Run a query, returning a list of rows.\"\"\"
    client = client or bigquery.Client()
    job_config = bigquery.QueryJobConfig(
        maximum_bytes_billed=max_bytes_billed,
        use_legacy_sql=False,
        query_parameters=[
            bigquery.ScalarQueryParameter(k, _sql_type(v), v)
            for k, v in (params or {}).items()
        ],
    )
    if label:
        job_config.labels = {"query_label": label[:63]}
    try:
        job = client.query(sql, job_config=job_config)
        rows = list(job.result())
        log.info("[%s] %s rows, %s bytes, cache_hit=%s",
                 label, len(rows), job.total_bytes_billed, job.cache_hit)
        return rows
    except GoogleCloudError:
        log.error("[%s] Query failed", label)
        raise


def dry_run(sql, params=None, client=None):
    \"\"\"Return (bytes, estimated_usd) without running the query.\"\"\"
    client = client or bigquery.Client()
    job_config = bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False, use_legacy_sql=False,
        query_parameters=[
            bigquery.ScalarQueryParameter(k, _sql_type(v), v)
            for k, v in (params or {}).items()
        ],
    )
    job = client.query(sql, job_config=job_config)
    total = job.total_bytes_processed or 0
    return total, (total / (1024**4)) * PRICE_PER_TIB
'''

with open("bq_query.py", "w") as f:
    f.write(QUERY_HELPER_SOURCE)

sys.path.insert(0, ".")
from bq_query import run, dry_run

print("✅ Wrote bq_query.py")
bytes_, cost = dry_run(sql)
print(f"   dry_run: {bytes_:,} bytes → ${cost:.6f}")

## 11. A cost-tracking wrapper

Log every query's cost into a BigQuery audit table. This is the first thing to
build when you take over a project.

In [ ]:
AUDIT_SCHEMA = [
    bigquery.SchemaField("query_id", "STRING"),
    bigquery.SchemaField("run_at", "TIMESTAMP"),
    bigquery.SchemaField("label", "STRING"),
    bigquery.SchemaField("bytes_billed", "INT64"),
    bigquery.SchemaField("est_cost_usd", "FLOAT64"),
    bigquery.SchemaField("cache_hit", "BOOL"),
]

audit_table = bigquery.Table(f"{PROJECT_ID}.monitoring.query_costs", schema=AUDIT_SCHEMA)
audit_table.time_partitioning = bigquery.TimePartitioning(
    type_=bigquery.TimePartitioningType.DAY, field="run_at"
)
audit_table = client.create_table(audit_table, exists_ok=True)
print(f"✓ {audit_table.full_table_id}")

In [ ]:
from datetime import datetime, timezone


def run_tracked(sql, label="", params=None):
    \"\"\"Run a query and record its cost in the audit table.\"\"\"
    # 1. Dry run
    est = estimate_cost(sql, params=params)

    # 2. Execute
    job = client.query(sql)
    rows = list(job.result())

    # 3. Record
    client.insert_rows_json(
        client.dataset("monitoring").table("query_costs"),
        [
            {
                "query_id": job.query_id,
                "run_at": datetime.now(timezone.utc).isoformat(),
                "label": label[:100],
                "bytes_billed": job.total_bytes_billed or 0,
                "est_cost_usd": est["estimated_cost_usd"],
                "cache_hit": job.cache_hit,
            }
        ],
    )
    return rows


run_tracked(sql, label="continent_counts")
run_tracked(sql, label="continent_counts")   # second run hits the cache

df = run_to_dataframe(f"SELECT * FROM `{PROJECT_ID}.monitoring.query_costs` ORDER BY run_at DESC LIMIT 10")
display(df)

## 🔑 Takeaways

- `client.query()` returns a `QueryJob`; **`.result()` blocks** and returns the rows.
- `.to_dataframe()` for exploration, `.pages` for large results.
- **Parameters always, with explicit types.**
- **`dry_run=True` + `total_bytes_processed`** is the pre-flight cost check.
- **`maximum_bytes_billed`** turns a cost accident into a failed job.
- `job.query_id` and `job.cache_hit` are your two best debugging fields.
- DML rewrites whole tables/partitions — prefer `MERGE` or partition replace.

---

## 📚 Next

[04 · Schema and Nested Data](./04-schema-and-nested-data.ipynb)